# DLGNet: Reproducible PII De-Identification Pipeline

This notebook implements the manuscript workflow without manually inserting manuscript results.  
Automatic generation of performance, ablation, comparative analysis, false-negative rate, and runtime evaluation tables.

Raw PII data are not included. Download the public Kaggle dataset separately.

In [ ]:
# Cell 1: Install pinned dependencies
!pip -q install \
    "torch>=2.1,<3" \
    "transformers>=4.41,<5" \
    "accelerate>=0.30,<2" \
    "evaluate>=0.4,<1" \
    "seqeval>=1.2,<2" \
    "sentencepiece>=0.2,<1" \
    "faker>=25,<40" \
    "pyyaml>=6,<7" \
    "scikit-learn>=1.3,<2" \
    "pandas>=2,<3" \
    "numpy>=1.24,<3"

In [ ]:
# Cell 2: Imports
import os
import re
import gc
import json
import math
import time
import yaml
import random
import hashlib
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
)

from faker import Faker

In [ ]:
# Cell 3: Central configuration
@dataclass
class Config:
    seed: int = 42
    data_dir: str = "/kaggle/input/pii-detection-removal-from-educational-data"
    output_dir: str = "/kaggle/working/dlgnet_reproducibility"

    detector_model: str = "microsoft/deberta-v3-base"
    llm_model: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

    max_length: int = 256
    stride: int = 64
    validation_ratio: float = 0.10

    learning_rate: float = 2e-5
    train_batch_size: int = 16
    eval_batch_size: int = 16
    epochs: int = 10
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    early_stopping_patience: int = 3

    llm_context_words: int = 64
    llm_max_new_tokens: int = 5

    gan_latent_dim: int = 100
    gan_max_name_len: int = 32
    gan_batch_size: int = 32
    gan_epochs: int = 2000
    gan_lr: float = 1e-4
    gan_beta1: float = 0.5
    gan_beta2: float = 0.9
    gan_lambda_gp: float = 10.0
    gan_critic_steps: int = 5

    runtime_documents: int = 50
    beta: float = 5.0

CFG = Config()
OUTPUT_DIR = Path(CFG.output_dir)
for subdir in ["models", "predictions", "tables", "prompts", "configs", "examples", "logs"]:
    (OUTPUT_DIR / subdir).mkdir(parents=True, exist_ok=True)

print(asdict(CFG))

In [ ]:
# Cell 4: Deterministic reproducibility settings
def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_global_seed(CFG.seed)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

reproducibility_manifest = {
    "seed": CFG.seed,
    "device": str(DEVICE),
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "deterministic_cudnn": bool(torch.backends.cudnn.deterministic)
        if torch.cuda.is_available() else None,
}
with open(OUTPUT_DIR / "configs" / "reproducibility_manifest.json", "w") as f:
    json.dump(reproducibility_manifest, f, indent=2)

print(json.dumps(reproducibility_manifest, indent=2))

In [ ]:
# Cell 5: Save machine-readable hyperparameter configuration
with open(OUTPUT_DIR / "configs" / "dlgnet.yaml", "w") as f:
    yaml.safe_dump(asdict(CFG), f, sort_keys=False)

print((OUTPUT_DIR / "configs" / "dlgnet.yaml").read_text())

In [ ]:
# Cell 6: Load the public Kaggle dataset
DATA_DIR = Path(CFG.data_dir)
TRAIN_PATH = DATA_DIR / "train.json"
TEST_PATH = DATA_DIR / "test.json"

if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        f"{TRAIN_PATH} was not found. Attach the public Kaggle dataset "
        "'PII Detection Removal from Educational Data'."
    )

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    raw_train_data = json.load(f)

raw_test_data = []
if TEST_PATH.exists():
    with open(TEST_PATH, "r", encoding="utf-8") as f:
        raw_test_data = json.load(f)

print("Annotated documents:", len(raw_train_data))
print("Public competition test documents:", len(raw_test_data))

In [ ]:
# Cell 7: Conservative text normalization and label consolidation
LABEL_MAP = {
    "B-NAME_STUDENT": "B-NAME",
    "I-NAME_STUDENT": "I-NAME",
    "B-EMAIL": "B-EMAIL",
    "I-EMAIL": "I-EMAIL",
    "B-PHONE_NUM": "B-PHONE",
    "I-PHONE_NUM": "I-PHONE",
    "B-ID_NUM": "B-ID",
    "I-ID_NUM": "I-ID",
    "B-STREET_ADDRESS": "B-ADDRESS",
    "I-STREET_ADDRESS": "I-ADDRESS",
    "O": "O",
}

RETAINED_LABELS = [
    "O",
    "B-NAME", "I-NAME",
    "B-EMAIL", "I-EMAIL",
    "B-PHONE", "I-PHONE",
    "B-ID", "I-ID",
    "B-ADDRESS", "I-ADDRESS",
]

def normalize_token(token: str) -> str:
    token = unicodedata.normalize("NFKC", str(token))
    token = re.sub(r"\s+", " ", token).strip()
    return token

def preprocess_document(doc: Dict[str, Any]) -> Dict[str, Any]:
    tokens = [normalize_token(token) for token in doc["tokens"]]
    labels = [LABEL_MAP.get(label, "O") for label in doc["labels"]]

    if len(tokens) != len(labels):
        raise ValueError("Token and label lengths are inconsistent.")

    return {
        "document": doc.get("document"),
        "tokens": tokens,
        "labels": labels,
    }

processed_data = [preprocess_document(doc) for doc in raw_train_data]

label_distribution = Counter(
    label for doc in processed_data for label in doc["labels"]
)
pd.DataFrame(
    sorted(label_distribution.items()),
    columns=["Label", "Token Count"]
)

In [ ]:
# Cell 8: BIO span extraction
def extract_spans(tokens: List[str], labels: List[str]) -> List[Dict[str, Any]]:
    if len(tokens) != len(labels):
        raise ValueError("tokens and labels must have equal lengths.")

    spans = []
    start = None
    entity_type = None

    def close_span(end_index: int) -> None:
        nonlocal start, entity_type
        if start is not None:
            spans.append({
                "start": start,
                "end": end_index,
                "type": entity_type,
                "text": " ".join(tokens[start:end_index + 1]),
            })
        start = None
        entity_type = None

    for index, label in enumerate(labels):
        if label == "O":
            close_span(index - 1)
            continue

        prefix, current_type = label.split("-", 1)

        if prefix == "B":
            close_span(index - 1)
            start = index
            entity_type = current_type
        elif prefix == "I":
            if start is None or entity_type != current_type:
                close_span(index - 1)
                start = index
                entity_type = current_type
        else:
            close_span(index - 1)

    close_span(len(labels) - 1)
    return spans

In [ ]:
# Cell 9: Dataset statistics generated from annotations
entity_counts = Counter()
entity_lengths = defaultdict(list)

for doc in processed_data:
    for span in extract_spans(doc["tokens"], doc["labels"]):
        entity_counts[span["type"]] += 1
        entity_lengths[span["type"]].append(span["end"] - span["start"] + 1)

total_entities = sum(entity_counts.values())

table1 = pd.DataFrame([
    {
        "Entity Type": entity_type,
        "Number of Entities": count,
        "Percentage (%)": 100.0 * count / total_entities,
    }
    for entity_type, count in sorted(entity_counts.items())
])

table2 = pd.DataFrame([
    {
        "Entity Type": entity_type,
        "Min Tokens": min(lengths),
        "Average Tokens": float(np.mean(lengths)),
        "Max Tokens": max(lengths),
    }
    for entity_type, lengths in sorted(entity_lengths.items())
])

display(table1)
display(table2)

In [ ]:
# Cell 10: Reproducible document-level train-validation split
rng = np.random.default_rng(CFG.seed)
indices = np.arange(len(processed_data))
rng.shuffle(indices)

validation_size = int(round(len(indices) * CFG.validation_ratio))
validation_indices = indices[:validation_size]
training_indices = indices[validation_size:]

train_docs = [processed_data[i] for i in training_indices]
val_docs = [processed_data[i] for i in validation_indices]

split_manifest = {
    "seed": CFG.seed,
    "validation_ratio": CFG.validation_ratio,
    "training_indices": training_indices.tolist(),
    "validation_indices": validation_indices.tolist(),
}

with open(OUTPUT_DIR / "configs" / "split_manifest.json", "w") as f:
    json.dump(split_manifest, f)

print("Training documents:", len(train_docs))
print("Validation documents:", len(val_docs))

In [ ]:
# Cell 11: Tokenizer and label mappings
tokenizer = AutoTokenizer.from_pretrained(CFG.detector_model, use_fast=True)

label2id = {label: index for index, label in enumerate(RETAINED_LABELS)}
id2label = {index: label for label, index in label2id.items()}
NUM_LABELS = len(RETAINED_LABELS)
O_ID = label2id["O"]

print(label2id)

In [ ]:
# Cell 12: Build overflow-aware subword windows
def build_window_records(
    documents: List[Dict[str, Any]],
    tokenizer,
    max_length: int,
    stride: int,
) -> List[Dict[str, Any]]:
    records = []

    for doc_index, doc in enumerate(tqdm(documents, desc="Tokenizing documents")):
        encoded = tokenizer(
            doc["tokens"],
            is_split_into_words=True,
            truncation=True,
            max_length=max_length,
            stride=stride,
            return_overflowing_tokens=True,
            padding=False,
        )

        for window_index in range(len(encoded["input_ids"])):
            word_ids = encoded.word_ids(batch_index=window_index)
            aligned_labels = []
            previous_word_id = None

            for word_id in word_ids:
                if word_id is None:
                    aligned_labels.append(-100)
                elif word_id != previous_word_id:
                    aligned_labels.append(label2id[doc["labels"][word_id]])
                else:
                    aligned_labels.append(-100)
                previous_word_id = word_id

            records.append({
                "input_ids": encoded["input_ids"][window_index],
                "attention_mask": encoded["attention_mask"][window_index],
                "labels": aligned_labels,
                "word_ids": [-1 if x is None else int(x) for x in word_ids],
                "document_index": doc_index,
                "window_index": window_index,
            })

    return records

train_window_records = build_window_records(
    train_docs, tokenizer, CFG.max_length, CFG.stride
)
val_window_records = build_window_records(
    val_docs, tokenizer, CFG.max_length, CFG.stride
)

print("Training windows:", len(train_window_records))
print("Validation windows:", len(val_window_records))

In [ ]:
# Cell 13: PyTorch window dataset
class TokenWindowDataset(Dataset):
    def __init__(self, records: List[Dict[str, Any]]):
        self.records = records

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:
        record = self.records[index]
        return {
            "input_ids": torch.tensor(record["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(record["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(record["labels"], dtype=torch.long),
        }

train_dataset = TokenWindowDataset(train_window_records)
val_dataset = TokenWindowDataset(val_window_records)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
# Cell 14: Compute class weights from training windows
valid_training_labels = [
    label_id
    for record in train_window_records
    for label_id in record["labels"]
    if label_id != -100
]

class_counts = np.bincount(valid_training_labels, minlength=NUM_LABELS)
class_weights = len(valid_training_labels) / (
    NUM_LABELS * np.maximum(class_counts, 1)
)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

class_weight_table = pd.DataFrame({
    "Label": [id2label[i] for i in range(NUM_LABELS)],
    "Count": class_counts,
    "Weight": class_weights.numpy(),
})
display(class_weight_table)

In [ ]:
# Cell 15: Privacy-oriented token metrics
def fbeta_from_precision_recall(
    precision: float, recall: float, beta: float
) -> float:
    denominator = beta**2 * precision + recall
    if denominator == 0:
        return 0.0
    return (1 + beta**2) * precision * recall / denominator

def flatten_valid_logits_and_labels(eval_prediction):
    logits, label_ids = eval_prediction
    mask = label_ids != -100
    return logits[mask], label_ids[mask]

def compute_training_metrics(eval_prediction) -> Dict[str, float]:
    logits, label_ids = flatten_valid_logits_and_labels(eval_prediction)
    predicted_ids = np.argmax(logits, axis=-1)

    true_binary = (label_ids != O_ID).astype(int)
    predicted_binary = (predicted_ids != O_ID).astype(int)

    probabilities = torch.softmax(
        torch.tensor(logits), dim=-1
    ).numpy()
    pii_probabilities = 1.0 - probabilities[:, O_ID]

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_binary,
        predicted_binary,
        average="binary",
        zero_division=0,
    )
    accuracy = accuracy_score(true_binary, predicted_binary)

    try:
        roc_auc = roc_auc_score(true_binary, pii_probabilities)
    except ValueError:
        roc_auc = float("nan")

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "fbeta_beta5": fbeta_from_precision_recall(
            float(precision), float(recall), CFG.beta
        ),
        "accuracy": float(accuracy),
        "roc_auc": float(roc_auc),
    }

In [ ]:
# Cell 16: Class-weighted Trainer
class WeightedTokenClassificationTrainer(Trainer):
    def __init__(self, *args, class_weights: torch.Tensor, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_function = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device),
            ignore_index=-100,
        )
        loss = loss_function(
            logits.view(-1, logits.shape[-1]),
            labels.view(-1),
        )

        return (loss, outputs) if return_outputs else loss

In [ ]:
# Cell 17: DeBERTa model and training arguments
detector_model = AutoModelForTokenClassification.from_pretrained(
    CFG.detector_model,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "models" / "deberta_checkpoints"),
    learning_rate=CFG.learning_rate,
    per_device_train_batch_size=CFG.train_batch_size,
    per_device_eval_batch_size=CFG.eval_batch_size,
    num_train_epochs=CFG.epochs,
    weight_decay=CFG.weight_decay,
    max_grad_norm=CFG.max_grad_norm,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_fbeta_beta5",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
    seed=CFG.seed,
    data_seed=CFG.seed,
    fp16=torch.cuda.is_available(),
)

trainer = WeightedTokenClassificationTrainer(
    model=detector_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_training_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=CFG.early_stopping_patience
        )
    ],
    class_weights=class_weights,
)

In [ ]:
# Cell 18: Train, evaluate, and save DeBERTa
train_result = trainer.train()
window_eval_results = trainer.evaluate()

DETECTOR_DIR = OUTPUT_DIR / "models" / "deberta_pii_detector"
trainer.save_model(DETECTOR_DIR)
tokenizer.save_pretrained(DETECTOR_DIR)

with open(OUTPUT_DIR / "logs" / "window_evaluation.json", "w") as f:
    json.dump(
        {
            key: float(value) if isinstance(value, (int, float, np.number)) else value
            for key, value in window_eval_results.items()
        },
        f,
        indent=2,
    )

window_eval_results

In [ ]:
# Cell 19: Full-document inference with overlapping-window probability aggregation
def infer_document(
    document: Dict[str, Any],
    model,
    tokenizer,
) -> Dict[str, Any]:
    tokens = document["tokens"]

    encoded = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=CFG.max_length,
        stride=CFG.stride,
        return_overflowing_tokens=True,
        padding=True,
        return_tensors="pt",
    )

    probability_sum = np.zeros((len(tokens), NUM_LABELS), dtype=np.float64)
    observation_count = np.zeros(len(tokens), dtype=np.int64)

    model.eval()
    with torch.inference_mode():
        for window_index in range(encoded["input_ids"].shape[0]):
            input_ids = encoded["input_ids"][window_index:window_index + 1].to(DEVICE)
            attention_mask = encoded["attention_mask"][window_index:window_index + 1].to(DEVICE)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            ).logits[0]
            probabilities = torch.softmax(logits, dim=-1).cpu().numpy()
            word_ids = encoded.word_ids(batch_index=window_index)

            seen_words = set()
            for token_position, word_id in enumerate(word_ids):
                if word_id is None or word_id in seen_words:
                    continue
                probability_sum[word_id] += probabilities[token_position]
                observation_count[word_id] += 1
                seen_words.add(word_id)

    if np.any(observation_count == 0):
        missing = np.where(observation_count == 0)[0].tolist()
        raise RuntimeError(f"Words missing from sliding-window inference: {missing[:10]}")

    averaged_probabilities = probability_sum / observation_count[:, None]
    predicted_ids = averaged_probabilities.argmax(axis=1)
    predicted_labels = [id2label[int(index)] for index in predicted_ids]

    return {
        "predicted_labels": predicted_labels,
        "probabilities": averaged_probabilities,
    }

detector_model = trainer.model.to(DEVICE)

In [ ]:
# Cell 20: Run and cache baseline DeBERTa predictions
baseline_predictions = []

for document_index, doc in enumerate(tqdm(val_docs, desc="DeBERTa validation inference")):
    inference = infer_document(doc, detector_model, tokenizer)
    baseline_predictions.append({
        "document_index": document_index,
        "true_labels": doc["labels"],
        "predicted_labels": inference["predicted_labels"],
        "pii_probabilities": (
            1.0 - inference["probabilities"][:, O_ID]
        ).tolist(),
    })

with open(OUTPUT_DIR / "predictions" / "deberta_validation.json", "w") as f:
    json.dump(baseline_predictions, f)

print("Saved", len(baseline_predictions), "document predictions.")

In [ ]:
# Cell 21: Metric computation from cached document predictions
def evaluate_prediction_records(
    prediction_records: List[Dict[str, Any]],
    prediction_key: str,
) -> Dict[str, float]:
    true_binary = []
    predicted_binary = []
    pii_probabilities = []

    for record in prediction_records:
        true_labels = record["true_labels"]
        predicted_labels = record[prediction_key]

        true_binary.extend([int(label != "O") for label in true_labels])
        predicted_binary.extend([int(label != "O") for label in predicted_labels])

        if prediction_key == "predicted_labels":
            pii_probabilities.extend(record["pii_probabilities"])
        else:
            # A hard semantic gate provides binary retained/rejected decisions.
            pii_probabilities.extend(
                [float(label != "O") for label in predicted_labels]
            )

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_binary,
        predicted_binary,
        average="binary",
        zero_division=0,
    )
    tn, fp, fn, tp = confusion_matrix(
        true_binary, predicted_binary, labels=[0, 1]
    ).ravel()

    try:
        roc_auc = roc_auc_score(true_binary, pii_probabilities)
    except ValueError:
        roc_auc = float("nan")

    return {
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "fbeta_beta5": fbeta_from_precision_recall(
            float(precision), float(recall), CFG.beta
        ),
        "accuracy": float(accuracy_score(true_binary, predicted_binary)),
        "roc_auc": float(roc_auc),
        "fnr": float(fn / (tp + fn)) if (tp + fn) else 0.0,
        "fpr": float(fp / (fp + tn)) if (fp + tn) else 0.0,
        "evaluated_tokens": int(len(true_binary)),
    }

baseline_metrics = evaluate_prediction_records(
    baseline_predictions, "predicted_labels"
)
baseline_metrics

In [ ]:
# Cell 22: Save the exact TinyLlama prompt template
PROMPT_TEMPLATE = """Given the following text, determine whether the marked span contains Personally Identifiable Information (PII) that should be de-identified.
Answer only YES or NO.

Text:
{context}

Question: Is the marked span a valid PII entity in this context?
Answer:"""

PROMPT_PATH = OUTPUT_DIR / "prompts" / "semantic_verification_prompt.txt"
PROMPT_PATH.write_text(PROMPT_TEMPLATE, encoding="utf-8")
print(PROMPT_PATH.read_text())

In [ ]:
# Cell 23: Load TinyLlama as a frozen inference-only semantic gate
llm_tokenizer = AutoTokenizer.from_pretrained(CFG.llm_model)

llm_model = AutoModelForCausalLM.from_pretrained(
    CFG.llm_model,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

llm_model.eval()
for parameter in llm_model.parameters():
    parameter.requires_grad = False

if llm_tokenizer.pad_token_id is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

print("TinyLlama loaded in inference-only mode.")

In [ ]:
# Cell 24: Deterministic NAME-span semantic verification
def build_semantic_context(
    tokens: List[str],
    start: int,
    end: int,
    window_words: int,
) -> str:
    left = max(0, start - window_words)
    right = min(len(tokens), end + window_words + 1)

    context_tokens = (
        tokens[left:start]
        + ["<entity>"]
        + tokens[start:end + 1]
        + ["</entity>"]
        + tokens[end + 1:right]
    )
    return " ".join(context_tokens)

def verify_name_span(
    tokens: List[str],
    start: int,
    end: int,
) -> Tuple[bool, str]:
    context = build_semantic_context(
        tokens, start, end, CFG.llm_context_words
    )
    prompt = PROMPT_TEMPLATE.format(context=context)

    if hasattr(llm_tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        input_ids = llm_tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
        )
        input_ids = input_ids.to(llm_model.device)
        attention_mask = torch.ones_like(input_ids)
    else:
        encoded = llm_tokenizer(prompt, return_tensors="pt")
        input_ids = encoded["input_ids"].to(llm_model.device)
        attention_mask = encoded["attention_mask"].to(llm_model.device)

    with torch.inference_mode():
        generated = llm_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=CFG.llm_max_new_tokens,
            do_sample=False,
            pad_token_id=llm_tokenizer.pad_token_id,
            eos_token_id=llm_tokenizer.eos_token_id,
        )

    new_tokens = generated[0, input_ids.shape[1]:]
    response = llm_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip().upper()

    if response.startswith("YES"):
        return True, response
    if response.startswith("NO"):
        return False, response

    # Privacy-oriented fallback: retain the DeBERTa prediction.
    return True, response

def apply_semantic_gate(
    tokens: List[str],
    predicted_labels: List[str],
) -> Tuple[List[str], List[Dict[str, Any]]]:
    refined_labels = predicted_labels.copy()
    decisions = []

    for span in extract_spans(tokens, predicted_labels):
        if span["type"] != "NAME":
            continue

        keep, raw_response = verify_name_span(
            tokens, span["start"], span["end"]
        )
        decisions.append({
            **span,
            "keep": bool(keep),
            "response": raw_response,
        })

        if not keep:
            for index in range(span["start"], span["end"] + 1):
                refined_labels[index] = "O"

    return refined_labels, decisions

In [ ]:
# Cell 25: Run and cache LLM-refined predictions
llm_predictions = []

for document_index, (doc, baseline) in enumerate(
    tqdm(
        zip(val_docs, baseline_predictions),
        total=len(val_docs),
        desc="TinyLlama semantic verification",
    )
):
    refined_labels, decisions = apply_semantic_gate(
        doc["tokens"], baseline["predicted_labels"]
    )

    llm_predictions.append({
        **baseline,
        "llm_refined_labels": refined_labels,
        "llm_decisions": decisions,
    })

with open(OUTPUT_DIR / "predictions" / "deberta_llm_validation.json", "w") as f:
    json.dump(llm_predictions, f)

llm_metrics = evaluate_prediction_records(
    llm_predictions, "llm_refined_labels"
)
llm_metrics

### Methodological interpretation

The LLM is a verification/filtering gate over DeBERTa-detected NAME spans.  
It can reduce false positives, but it cannot recover a span that DeBERTa never detected. Therefore, the executable results may show unchanged or reduced recall after semantic verification. The code does not manually force an FNR improvement.

In [ ]:
# Cell 26: Extract training NAME spans for offline WGAN-GP training
real_names = []

for doc in train_docs:
    for span in extract_spans(doc["tokens"], doc["labels"]):
        if span["type"] == "NAME" and span["text"].strip():
            real_names.append(span["text"].strip())

real_names = sorted(set(real_names))
print("Unique training NAME spans:", len(real_names))
print("SHA-256 digest of name list:", hashlib.sha256(
    "\n".join(real_names).encode("utf-8")
).hexdigest())

# Do not save or publish the extracted raw names.

In [ ]:
# Cell 27: Character representation for WGAN-GP
GAN_VOCAB = list(
    " abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ.'-"
)
char2id = {character: index for index, character in enumerate(GAN_VOCAB)}
id2char = {index: character for character, index in char2id.items()}

GAN_VOCAB_SIZE = len(GAN_VOCAB)
GAN_OUTPUT_DIM = CFG.gan_max_name_len * GAN_VOCAB_SIZE

def encode_name(name: str) -> np.ndarray:
    normalized = unicodedata.normalize("NFKC", name)
    normalized = normalized[:CFG.gan_max_name_len].ljust(CFG.gan_max_name_len)

    encoded = np.zeros(
        (CFG.gan_max_name_len, GAN_VOCAB_SIZE),
        dtype=np.float32,
    )
    for position, character in enumerate(normalized):
        encoded[position, char2id.get(character, char2id[" "])] = 1.0

    return encoded.reshape(-1)

def decode_name(vector: np.ndarray) -> str:
    character_matrix = vector.reshape(
        CFG.gan_max_name_len, GAN_VOCAB_SIZE
    )
    name = "".join(
        id2char[int(np.argmax(character_distribution))]
        for character_distribution in character_matrix
    )
    return re.sub(r"\s+", " ", name).strip(" .'-")

In [ ]:
# Cell 28: WGAN-GP generator, critic, and gradient penalty
class NameGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(CFG.gan_latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, GAN_OUTPUT_DIM),
        )

    def forward(self, noise):
        logits = self.network(noise)
        logits = logits.view(
            -1, CFG.gan_max_name_len, GAN_VOCAB_SIZE
        )
        probabilities = torch.softmax(logits, dim=-1)
        return probabilities.view(-1, GAN_OUTPUT_DIM)

class NameCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(GAN_OUTPUT_DIM, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
        )

    def forward(self, encoded_names):
        return self.network(encoded_names)

def compute_gradient_penalty(
    critic: nn.Module,
    real_samples: torch.Tensor,
    fake_samples: torch.Tensor,
) -> torch.Tensor:
    batch_size = real_samples.shape[0]
    epsilon = torch.rand(
        batch_size, 1, device=real_samples.device
    ).expand_as(real_samples)

    interpolated = (
        epsilon * real_samples
        + (1.0 - epsilon) * fake_samples
    )
    interpolated.requires_grad_(True)

    critic_scores = critic(interpolated)
    gradients = torch.autograd.grad(
        outputs=critic_scores,
        inputs=interpolated,
        grad_outputs=torch.ones_like(critic_scores),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]

    gradients = gradients.view(batch_size, -1)
    return ((gradients.norm(2, dim=1) - 1.0) ** 2).mean()

In [ ]:
# Cell 29: Train WGAN-GP for 2,000 epochs and save logs/checkpoint
encoded_name_array = np.stack(
    [encode_name(name) for name in real_names]
)
name_dataset = torch.tensor(
    encoded_name_array, dtype=torch.float32
)
name_loader = DataLoader(
    name_dataset,
    batch_size=CFG.gan_batch_size,
    shuffle=True,
    drop_last=len(name_dataset) >= CFG.gan_batch_size,
    generator=torch.Generator().manual_seed(CFG.seed),
)

generator = NameGenerator().to(DEVICE)
critic = NameCritic().to(DEVICE)

generator_optimizer = torch.optim.Adam(
    generator.parameters(),
    lr=CFG.gan_lr,
    betas=(CFG.gan_beta1, CFG.gan_beta2),
)
critic_optimizer = torch.optim.Adam(
    critic.parameters(),
    lr=CFG.gan_lr,
    betas=(CFG.gan_beta1, CFG.gan_beta2),
)

gan_history = []

for epoch in range(1, CFG.gan_epochs + 1):
    epoch_critic_losses = []
    epoch_generator_losses = []
    epoch_penalties = []

    for real_batch in name_loader:
        real_batch = real_batch.to(DEVICE)
        batch_size = real_batch.shape[0]

        for _ in range(CFG.gan_critic_steps):
            noise = torch.randn(
                batch_size, CFG.gan_latent_dim, device=DEVICE
            )
            fake_batch = generator(noise).detach()

            real_score = critic(real_batch).mean()
            fake_score = critic(fake_batch).mean()
            gradient_penalty = compute_gradient_penalty(
                critic, real_batch, fake_batch
            )

            critic_loss = (
                fake_score
                - real_score
                + CFG.gan_lambda_gp * gradient_penalty
            )

            critic_optimizer.zero_grad(set_to_none=True)
            critic_loss.backward()
            critic_optimizer.step()

        noise = torch.randn(
            batch_size, CFG.gan_latent_dim, device=DEVICE
        )
        generated_batch = generator(noise)
        generator_loss = -critic(generated_batch).mean()

        generator_optimizer.zero_grad(set_to_none=True)
        generator_loss.backward()
        generator_optimizer.step()

        epoch_critic_losses.append(float(critic_loss.detach().cpu()))
        epoch_generator_losses.append(float(generator_loss.detach().cpu()))
        epoch_penalties.append(float(gradient_penalty.detach().cpu()))

    record = {
        "epoch": epoch,
        "critic_loss": float(np.mean(epoch_critic_losses)),
        "generator_loss": float(np.mean(epoch_generator_losses)),
        "gradient_penalty": float(np.mean(epoch_penalties)),
    }
    gan_history.append(record)

    if epoch % 100 == 0 or epoch > CFG.gan_epochs - 5:
        print(record)

gan_history_df = pd.DataFrame(gan_history)
gan_history_df.to_csv(
    OUTPUT_DIR / "logs" / "wgan_gp_training.csv",
    index=False,
)

torch.save(
    {
        "generator_state_dict": generator.state_dict(),
        "critic_state_dict": critic.state_dict(),
        "config": asdict(CFG),
        "vocab": GAN_VOCAB,
    },
    OUTPUT_DIR / "models" / "wgan_gp_names.pt",
)

In [ ]:
# Cell 30: Privacy-safe synthetic name generation
faker = Faker()
faker.seed_instance(CFG.seed)

def is_plausible_generated_name(name: str) -> bool:
    if not (2 <= len(name) <= CFG.gan_max_name_len):
        return False
    if not re.fullmatch(r"[A-Za-z][A-Za-z .'-]*", name):
        return False
    alphabetic_count = sum(character.isalpha() for character in name)
    return alphabetic_count >= 2

def generate_wgan_name(max_attempts: int = 20) -> str:
    generator.eval()

    with torch.inference_mode():
        for _ in range(max_attempts):
            noise = torch.randn(
                1, CFG.gan_latent_dim, device=DEVICE
            )
            generated = generator(noise).cpu().numpy()[0]
            candidate = decode_name(generated)

            if is_plausible_generated_name(candidate):
                return candidate

    # Documented fail-safe for malformed GAN output.
    return faker.name()

[generate_wgan_name() for _ in range(5)]

In [ ]:
# Cell 31: Type-aware replacement with stable document-level mapping
def generate_email_surrogate() -> str:
    return f"{faker.user_name()}@example.com"

def generate_phone_surrogate(original: str) -> str:
    replacement_digits = iter(
        random.choices(
            "0123456789",
            k=sum(character.isdigit() for character in original),
        )
    )
    return "".join(
        next(replacement_digits) if character.isdigit() else character
        for character in original
    )

def generate_id_surrogate(original: str) -> str:
    def replacement_character(character: str) -> str:
        if character.isdigit():
            return random.choice("0123456789")
        if character.isalpha():
            return random.choice("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
        return character

    transformed = "".join(
        replacement_character(character) for character in original
    )
    return transformed if transformed else "ID-" + "".join(
        random.choices("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789", k=8)
    )

def generate_address_surrogate() -> str:
    return "[ADDRESS]"

def replacement_for_entity(
    entity_text: str,
    entity_type: str,
    mapping: Dict[Tuple[str, str], str],
    use_gan: bool,
) -> str:
    key = (entity_text, entity_type)

    if key in mapping:
        return mapping[key]

    if entity_type == "NAME":
        replacement = (
            generate_wgan_name() if use_gan else faker.name()
        )
    elif entity_type == "EMAIL":
        replacement = generate_email_surrogate()
    elif entity_type == "PHONE":
        replacement = generate_phone_surrogate(entity_text)
    elif entity_type == "ID":
        replacement = generate_id_surrogate(entity_text)
    elif entity_type == "ADDRESS":
        replacement = generate_address_surrogate()
    else:
        replacement = "[PII]"

    mapping[key] = replacement
    return replacement

In [ ]:
# Cell 32: Reconstruct a de-identified document from supplied predictions
def reconstruct_document(
    tokens: List[str],
    labels: List[str],
    use_gan: bool,
) -> Dict[str, Any]:
    spans = extract_spans(tokens, labels)
    mapping = {}
    output_tokens = []
    current_index = 0

    for span in spans:
        while current_index < span["start"]:
            output_tokens.append(tokens[current_index])
            current_index += 1

        replacement = replacement_for_entity(
            span["text"],
            span["type"],
            mapping,
            use_gan=use_gan,
        )
        output_tokens.extend(replacement.split())
        current_index = span["end"] + 1

    while current_index < len(tokens):
        output_tokens.append(tokens[current_index])
        current_index += 1

    return {
        "tokens": output_tokens,
        "text": " ".join(output_tokens),
        "mapping": {
            f"{entity_type}:{entity_text}": surrogate
            for (entity_text, entity_type), surrogate in mapping.items()
        },
    }

In [ ]:
# Cell 33: Utility metrics
def length_ratio(
    original_tokens: List[str],
    deidentified_tokens: List[str],
) -> float:
    if not original_tokens:
        return 0.0
    return len(deidentified_tokens) / len(original_tokens)

def multiset_token_overlap(
    original_tokens: List[str],
    deidentified_tokens: List[str],
) -> float:
    if not original_tokens:
        return 0.0

    original_counter = Counter(original_tokens)
    deidentified_counter = Counter(deidentified_tokens)
    shared_count = sum(
        min(count, deidentified_counter[token])
        for token, count in original_counter.items()
    )
    return shared_count / len(original_tokens)

In [ ]:
# Cell 34: Residual PII check by reapplying DeBERTa
def predict_unlabelled_tokens(tokens: List[str]) -> List[str]:
    dummy_document = {
        "tokens": tokens,
        "labels": ["O"] * len(tokens),
    }
    return infer_document(
        dummy_document, detector_model, tokenizer
    )["predicted_labels"]

def evaluate_reconstruction_variant(
    prediction_records: List[Dict[str, Any]],
    prediction_key: str,
    use_gan: bool,
) -> Dict[str, Any]:
    rows = []
    total_residual_tokens = 0
    total_output_tokens = 0

    for doc, record in tqdm(
        zip(val_docs, prediction_records),
        total=len(val_docs),
        desc=f"Replacement evaluation: {prediction_key}, GAN={use_gan}",
    ):
        reconstructed = reconstruct_document(
            doc["tokens"],
            record[prediction_key],
            use_gan=use_gan,
        )

        residual_labels = predict_unlabelled_tokens(
            reconstructed["tokens"]
        )
        residual_count = sum(label != "O" for label in residual_labels)

        total_residual_tokens += residual_count
        total_output_tokens += len(reconstructed["tokens"])

        rows.append({
            "document": doc.get("document"),
            "length_ratio": length_ratio(
                doc["tokens"], reconstructed["tokens"]
            ),
            "token_overlap": multiset_token_overlap(
                doc["tokens"], reconstructed["tokens"]
            ),
            "residual_pii_tokens": residual_count,
            "output_tokens": len(reconstructed["tokens"]),
        })

    detail_df = pd.DataFrame(rows)

    return {
        "mean_length_ratio": float(detail_df["length_ratio"].mean()),
        "mean_token_overlap": float(detail_df["token_overlap"].mean()),
        "residual_pii_tokens": int(total_residual_tokens),
        "residual_pii_rate": (
            float(total_residual_tokens / total_output_tokens)
            if total_output_tokens else 0.0
        ),
        "details": detail_df,
    }

In [ ]:
# Cell 35: Evaluate no-GAN and full DLGNet reconstruction variants
no_gan_utility = evaluate_reconstruction_variant(
    baseline_predictions,
    prediction_key="predicted_labels",
    use_gan=False,
)

full_dlgnet_utility = evaluate_reconstruction_variant(
    llm_predictions,
    prediction_key="llm_refined_labels",
    use_gan=True,
)

no_gan_utility["details"].to_csv(
    OUTPUT_DIR / "predictions" / "no_gan_utility_by_document.csv",
    index=False,
)
full_dlgnet_utility["details"].to_csv(
    OUTPUT_DIR / "predictions" / "full_dlgnet_utility_by_document.csv",
    index=False,
)

{
    "no_gan": {k: v for k, v in no_gan_utility.items() if k != "details"},
    "full_dlgnet": {
        k: v for k, v in full_dlgnet_utility.items() if k != "details"
    },
}

### Ablation logic used in the executable tables

- **DeBERTa Only:** baseline detector predictions.
- **DeBERTa + LLM:** baseline predictions after NAME-span filtering.
- **DeBERTa + Replacement (No GAN):** detector predictions are unchanged; rule/Faker replacements are evaluated through utility and residual-PII measures.
- **Full DLGNet:** LLM-refined predictions plus WGAN-GP/rule-based replacement.

Replacement cannot retroactively change the detector's token-level Precision, Recall, F1, or Fβ. The notebook therefore does not fabricate different detection metrics for replacement-only variants.

In [ ]:
# Cell 36: Generate Table 4 from full DLGNet detection predictions
table4 = pd.DataFrame([
    {"Metric": "Precision", "Value": llm_metrics["precision"]},
    {"Metric": "Recall", "Value": llm_metrics["recall"]},
    {"Metric": "F1-Score", "Value": llm_metrics["f1"]},
    {
        "Metric": f"micro-Fβ (β = {int(CFG.beta)})",
        "Value": llm_metrics["fbeta_beta5"],
    },
    {"Metric": "Accuracy", "Value": llm_metrics["accuracy"]},
    {"Metric": "ROC-AUC", "Value": llm_metrics["roc_auc"]},
    {
        "Metric": "Final Validation Loss",
        "Value": float(window_eval_results["eval_loss"]),
    },
])

table4["Value (%)"] = np.where(
    table4["Metric"].eq("Final Validation Loss"),
    table4["Value"],
    table4["Value"] * 100.0,
)
table4 = table4.drop(columns=["Value"])
display(table4)

In [ ]:
# Cell 37: Generate Table 10 from computed FNR values
table10_rows = [
    {
        "Model Configuration": "DeBERTa Only",
        "False Negative Rate (%)": baseline_metrics["fnr"] * 100.0,
    },
    {
        "Model Configuration": "DeBERTa + LLM",
        "False Negative Rate (%)": llm_metrics["fnr"] * 100.0,
    },
    {
        "Model Configuration": "Full DLGNet",
        # Replacement does not alter detection labels.
        "False Negative Rate (%)": llm_metrics["fnr"] * 100.0,
    },
]

table10 = pd.DataFrame(table10_rows)
baseline_fnr = table10.loc[
    table10["Model Configuration"].eq("DeBERTa Only"),
    "False Negative Rate (%)",
].iloc[0]

table10["Relative Change vs. DeBERTa Only (%)"] = (
    100.0
    * (
        baseline_fnr
        - table10["False Negative Rate (%)"]
    )
    / baseline_fnr
    if baseline_fnr != 0
    else np.nan
)

display(table10)

In [ ]:
# Cell 38: Generate Table 11 without manually entered results
def metric_row(
    name: str,
    metrics: Dict[str, float],
    utility: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    return {
        "Model Configuration": name,
        "Precision (%)": metrics["precision"] * 100.0,
        "Recall (%)": metrics["recall"] * 100.0,
        "F1-Score (%)": metrics["f1"] * 100.0,
        f"micro-Fβ (β = {int(CFG.beta)}) (%)":
            metrics["fbeta_beta5"] * 100.0,
        "Length Ratio": (
            utility["mean_length_ratio"] if utility else np.nan
        ),
        "Token Overlap": (
            utility["mean_token_overlap"] if utility else np.nan
        ),
        "Residual PII Rate (%)": (
            utility["residual_pii_rate"] * 100.0
            if utility else np.nan
        ),
    }

table11 = pd.DataFrame([
    metric_row("DeBERTa Only", baseline_metrics),
    metric_row("DeBERTa + LLM", llm_metrics),
    metric_row(
        "DeBERTa + Replacement (No GAN)",
        baseline_metrics,
        no_gan_utility,
    ),
    metric_row(
        "Full DLGNet (Proposed)",
        llm_metrics,
        full_dlgnet_utility,
    ),
])

display(table11)

In [ ]:
# Cell 39: Generate revised Table 12 comparative analysis
comparison_metrics = [
    ("Precision", "precision"),
    ("Recall", "recall"),
    ("F1-Score", "f1"),
    (f"micro-Fβ (β = {int(CFG.beta)})", "fbeta_beta5"),
]

table12 = pd.DataFrame([
    {
        "Comparison": "DeBERTa vs DeBERTa + LLM",
        "Metric": display_name,
        "DeBERTa Only (%)": baseline_metrics[key] * 100.0,
        "DeBERTa + LLM (%)": llm_metrics[key] * 100.0,
        "Absolute Difference (percentage points)":
            (llm_metrics[key] - baseline_metrics[key]) * 100.0,
    }
    for display_name, key in comparison_metrics
])

display(table12)

In [ ]:
# Cell 40: Component-level runtime measurement
def synchronize_device() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize()

runtime_records = []

for doc, baseline in tqdm(
    list(zip(val_docs, baseline_predictions))[:CFG.runtime_documents],
    desc="Runtime benchmarking",
):
    synchronize_device()
    start = time.perf_counter()
    fresh_baseline = infer_document(doc, detector_model, tokenizer)
    synchronize_device()
    detector_ms = (time.perf_counter() - start) * 1000.0

    synchronize_device()
    start = time.perf_counter()
    refined_labels, _ = apply_semantic_gate(
        doc["tokens"], fresh_baseline["predicted_labels"]
    )
    synchronize_device()
    llm_ms = (time.perf_counter() - start) * 1000.0

    start = time.perf_counter()
    spans = extract_spans(doc["tokens"], refined_labels)
    span_ms = (time.perf_counter() - start) * 1000.0

    synchronize_device()
    start = time.perf_counter()
    _ = reconstruct_document(
        doc["tokens"], refined_labels, use_gan=True
    )
    synchronize_device()
    replacement_ms = (time.perf_counter() - start) * 1000.0

    runtime_records.append({
        "DeBERTa Token-Level Detection": detector_ms,
        "LLM Semantic Verification": llm_ms,
        "PII Span Aggregation": span_ms,
        "Generative / Rule-Based Replacement": replacement_ms,
    })

runtime_df = pd.DataFrame(runtime_records)
runtime_df.to_csv(
    OUTPUT_DIR / "logs" / "runtime_by_document.csv",
    index=False,
)

component_means = runtime_df.mean()
total_runtime = float(component_means.sum())

table13 = pd.DataFrame({
    "Component": list(component_means.index)
        + ["Total DLGNet Inference Time"],
    "Average Inference Time (ms/document)":
        list(component_means.values) + [total_runtime],
})

table13["Relative Contribution (%)"] = (
    100.0
    * table13["Average Inference Time (ms/document)"]
    / total_runtime
)
table13.loc[
    table13["Component"].eq("Total DLGNet Inference Time"),
    "Relative Contribution (%)",
] = 100.0

display(table13)

In [ ]:
# Cell 41: Synthetic examples only; no raw dataset examples are exported
synthetic_examples = [
    {
        "tokens": [
            "Student", "Aarav", "Mehta", "can", "be", "contacted",
            "at", "aarav.mehta@example.edu", "."
        ],
        "labels": [
            "O", "B-NAME", "I-NAME", "O", "O", "O",
            "O", "B-EMAIL", "O"
        ],
    },
    {
        "tokens": [
            "The", "identifier", "AB-10482", "is", "associated",
            "with", "21", "Park", "Street", "."
        ],
        "labels": [
            "O", "O", "B-ID", "O", "O",
            "O", "B-ADDRESS", "I-ADDRESS", "I-ADDRESS", "O"
        ],
    },
]

synthetic_outputs = []
for example in synthetic_examples:
    reconstructed = reconstruct_document(
        example["tokens"], example["labels"], use_gan=True
    )
    synthetic_outputs.append({
        "original_text": " ".join(example["tokens"]),
        "deidentified_text": reconstructed["text"],
    })

with open(
    OUTPUT_DIR / "examples" / "synthetic_examples.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(synthetic_outputs, f, indent=2)

pd.DataFrame(synthetic_outputs)

In [ ]:
# Cell 42: Save all reproducibility tables and machine-readable metrics
tables = {
    "table1_entity_distribution.csv": table1,
    "table2_token_length_statistics.csv": table2,
    "table4_token_level_performance.csv": table4,
    "table10_fnr_progression.csv": table10,
    "table11_ablation_study.csv": table11,
    "table12_comparative_performance.csv": table12,
    "table13_runtime_analysis.csv": table13,
}

for filename, dataframe in tables.items():
    dataframe.to_csv(
        OUTPUT_DIR / "tables" / filename,
        index=False,
    )

summary_metrics = {
    "deberta_only": baseline_metrics,
    "deberta_plus_llm": llm_metrics,
    "no_gan_utility": {
        key: value
        for key, value in no_gan_utility.items()
        if key != "details"
    },
    "full_dlgnet_utility": {
        key: value
        for key, value in full_dlgnet_utility.items()
        if key != "details"
    },
}

with open(
    OUTPUT_DIR / "predictions" / "summary_metrics.json",
    "w",
) as f:
    json.dump(summary_metrics, f, indent=2)

print("Saved tables to:", OUTPUT_DIR / "tables")

In [ ]:
# Cell 43: Generate requirements.txt and README.md for GitHub
requirements = """torch>=2.1,<3
transformers>=4.41,<5
accelerate>=0.30,<2
evaluate>=0.4,<1
seqeval>=1.2,<2
sentencepiece>=0.2,<1
faker>=25,<40
pyyaml>=6,<7
scikit-learn>=1.3,<2
pandas>=2,<3
numpy>=1.24,<3
"""

(OUTPUT_DIR / "requirements.txt").write_text(
    requirements, encoding="utf-8"
)

readme = f"""# DLGNet PII De-Identification

Official reproducibility implementation of DLGNet.

## Public dataset

Download the public Kaggle competition dataset:
`PII Detection Removal from Educational Data`

The repository does not redistribute `train.json`, `test.json`, raw PII,
or extracted training names.

Expected Kaggle path:

```text
{CFG.data_dir}
```

## Reproducibility materials

The release includes:

- conservative preprocessing and five-category BIO consolidation;
- deterministic split manifest and random seed;
- class-weighted DeBERTa training;
- TinyLlama prompt template and inference-only verification;
- WGAN-GP training for NAME surrogates;
- stable type-aware replacement;
- residual PII and utility evaluation;
- scripts generating Tables 4, 10, 11, 12, and 13 from model outputs.

## Run

Execute `DLGNet_Reproducibility.ipynb` from top to bottom in Kaggle.

## Important interpretation

The LLM is a post-hoc filtering gate for detected NAME spans. It can
reduce false positives but cannot recover a completely missed span.
Replacement does not change token-level detector metrics. Tables are
therefore generated from actual stage outputs rather than hard-coded
manuscript values.

## Outputs

Generated files are written to:

```text
{CFG.output_dir}
```

## Privacy

Only synthetic demonstration examples are included in the repository.
"""

(OUTPUT_DIR / "README.md").write_text(readme, encoding="utf-8")
print(readme)

In [ ]:
# Cell 44: Create a publication-ready ZIP for GitHub upload
import shutil

archive_path = shutil.make_archive(
    "/kaggle/working/DLGNet_GitHub_Release",
    "zip",
    root_dir=OUTPUT_DIR,
)

print("Created:", archive_path)